# EDA Behavior

Exploratory analysis for behavior datasets (online shoppers + CERT r4.2).

Steps:
- Verify data presence and sizes.
- Inspect label balance and categorical distributions.
- Sample CERT r4.2 event logs.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
import pandas as pd
from pathlib import Path

behavior_root = REPO_ROOT / 'data' / 'raw' / 'behavior'
online_path = behavior_root / 'online_shoppers_intention.csv'
cert_root = behavior_root / 'r4.2'

summary = {
    'online_shoppers': {},
    'cert_r4_2': {},
}

print('Behavior root:', behavior_root)
if not behavior_root.exists():
    print('Missing:', behavior_root)
else:
    for child in sorted(behavior_root.iterdir()):
        kind = 'dir' if child.is_dir() else 'file'
        print(f' - {child.name} ({kind})')

if online_path.exists():
    summary['online_shoppers']['path'] = str(online_path)
    summary['online_shoppers']['size_mb'] = round(online_path.stat().st_size / 1024**2, 2)
else:
    print('Missing online_shoppers_intention.csv:', online_path)

if cert_root.exists():
    summary['cert_r4_2']['path'] = str(cert_root)
    files = [p for p in cert_root.iterdir() if p.is_file()]
    summary['cert_r4_2']['file_count'] = len(files)
    for sample in files[:8]:
        print('CERT file:', sample.name)
else:
    print('Missing CERT r4.2 directory:', cert_root)


In [ ]:
# Online shoppers dataset overview.
if online_path.exists():
    df = pd.read_csv(online_path)
    summary['online_shoppers']['rows'] = int(df.shape[0])
    summary['online_shoppers']['cols'] = int(df.shape[1])
    print('online_shoppers_intention.csv shape:', df.shape)
    print('Columns:', list(df.columns))

    missing = df.isna().sum().sort_values(ascending=False)
    print('Missing values (top 10):')
    print(missing.head(10))

    dupes = int(df.duplicated().sum())
    summary['online_shoppers']['duplicates'] = dupes
    print('Duplicate rows:', dupes)

    if 'Revenue' in df.columns:
        counts = df['Revenue'].value_counts().to_dict()
        summary['online_shoppers']['label_counts'] = counts
        print('Revenue label distribution:', counts)

    for col in ['VisitorType', 'Month', 'Region', 'TrafficType']:
        if col in df.columns:
            top_vals = df[col].value_counts().head(8).to_dict()
            summary['online_shoppers'][f'{col}_top'] = top_vals
            print(f'Top {col}:', top_vals)


In [ ]:
# Sample CERT r4.2 event logs.
if cert_root.exists():
    sample_files = ['logon.csv', 'email.csv', 'file.csv', 'device.csv']
    cert_summary = {}
    for name in sample_files:
        path = cert_root / name
        if not path.exists():
            continue
        print('')
        print(f'Preview {name}')
        df = pd.read_csv(path, nrows=50000, low_memory=False)
        cert_summary[name] = {
            'rows_sampled': int(df.shape[0]),
            'cols': int(df.shape[1]),
            'columns': list(df.columns),
        }
        print('Shape (sample):', df.shape)
        print('Columns:', list(df.columns))
        for col in ['user', 'user_id', 'pc', 'activity']:
            if col in df.columns:
                top_vals = df[col].value_counts().head(5).to_dict()
                cert_summary[name][f'{col}_top'] = top_vals
                print(f'Top {col}:', top_vals)
    summary['cert_r4_2']['samples'] = cert_summary


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_behavior_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize behavior-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'behavior' in str(item.get('name', '')).lower()
    ]
    if not items:
        print('No behavior entries found in TRAINING_DATA.json')
    else:
        print('behavior datasets in audit:')
        for item in items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
